# We will visualize the new june data

In [21]:
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted


rows = list(string.ascii_uppercase[:16])  # A-P
cols = list(range(1, 25))                 # 1-24

wells = [f"{r}{c:02d}" for r in rows for c in cols]

raw_dataset = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                   "Study/enVision/enVision_formatted/2026-06-01EnVision"
                   "_20xRep._.FORMATTED.tsv")

detrended_datast = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                        "Study/enVision/enVision_preprocessed/2026-06-01._.20"
                        "xREP._.PREPROCESSED.parquet")

# plate_map = (
#     pl.read_excel(
#         "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#         "Study/platemaps/2026_05_26_JUNE-EXP_plate_map.xlsx",
#         engine='openpyxl'
#     )
#     .select(["384_Well", 
#              "MoleculeID", 
#              "Human_Readable_Name",
#              ])
#     .unique(subset=["384_Well"])
#     .with_columns(
#         pl.col("MoleculeID").str.strip_chars()
#     )
# )

# period_results = (
#     pl.read_csv("/home/derfelt/PartchLabFiles/Circadian "
#                 "Rhythms/Live Study/enVision/period_dete"
#                 "ction_results/6-01_per-well_period_results.csv")
# )

# plate_map = plate_map.join(period_results, left_on="384_Well", right_on="Well", how="right")

# plate_map.write_excel("/home/derfelt/PartchLabFiles/Circadian "
#                       "Rhythms/Live Study/platemaps/"
#                       "2026_05_26_JUNE-EXP_plate_map._.W-PERIODS.xlsx")

plate_map = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026_05_26"
                  "_JUNE-EXP_plate_map._.W-PERIODS.xlsx", engine='openpyxl')
)

display(plate_map)

MoleculeID,Human_Readable_Name,Well,group_label,detected_period
str,str,str,str,f64
"""Chroma_DeepRed_Nucleo_ER_bmal1""","""no_treatment""","""F09""","""Chroma_DeepRed_Nucleo_ER_bmal1…",47.666667
"""Chroma_DeepRed_Nucleo_ER_bmal1""","""no_treatment""","""F07""","""Chroma_DeepRed_Nucleo_ER_bmal1…",47.666667
"""Chroma_DeepRed_Nucleo_ER_bmal1""","""no_treatment""","""F08""","""Chroma_DeepRed_Nucleo_ER_bmal1…",26.0
"""Chroma_DeepRed_Nucleo_ER_per2""","""no_treatment""","""J09""","""Chroma_DeepRed_Nucleo_ER_per2 …",28.6
"""Chroma_DeepRed_Nucleo_ER_per2""","""no_treatment""","""J07""","""Chroma_DeepRed_Nucleo_ER_per2 …",28.6
…,…,…,…,…
null,null,"""E23""","""nan | nan""",47.666667
null,null,"""F04""","""nan | nan""",47.666667
null,null,"""I06""","""nan | nan""",40.857143


In [56]:
def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

def savgol_smooth(
    dataset: pl.DataFrame, 
    value_col: str,
    well_col: str,
    window_length: int = 13, 
    polyorder: int = 3
    ) -> pl.DataFrame:
    return (
        dataset
        .group_by(well_col)
        .map_groups(
            lambda g: 
                g.with_columns(
                    pl.Series(value_col, savgol_filter(
                        g[value_col].to_numpy(),
                        window_length=window_length,
                        polyorder=polyorder
                    ))
                )
        )
    )

def plot_plates(
    timeseries: pl.DataFrame,
    timepoint_col: str,
    well_col: str,
    grouping_col: str,
    value_col:str,
    meta_cols: list[str],
    out_file: Path,
    smoothed: bool = True,
    col_wrap: int = 12,
    tick_interval: int = 12) -> None:
    
    if smoothed:
        timeseries = savgol_smooth(
            dataset=timeseries,
            value_col=value_col,
            well_col=well_col
        )
    
    with PdfPages(str(out_file)) as pdf:
        g = sns.FacetGrid(
            data=timeseries,
            col=grouping_col,
            col_wrap=col_wrap,
            height=2.5,
            aspect=1.8,
            sharey=False,
            sharex=True,
        )
        
        g.map_dataframe(
            sns.lineplot,
            x=timepoint_col,
            y=value_col,
            color='red',
            linewidth=1.5
        )

        x_min = timeseries[timepoint_col].min()
        x_max = timeseries[timepoint_col].max()
        
        tick_marks = np.arange(np.ceil(x_min / tick_interval) * tick_interval, x_max, tick_interval)

        for ax in g.axes.flat:
            ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
            ax.set_xticks(tick_marks)
            ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
            for mark in tick_marks:
                ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        g.set_titles(col_template='Group: {col_name}')
        g.set_axis_labels('Time (hours)', 'Detrended signal')
        g.tight_layout()
        pdf.savefig(g.figure)
        plt.close()
    
        groups = timeseries[grouping_col].unique().to_list()
        for group in tqdm(groups, desc='Well progress', total=len(groups)):
            group_df = (
                timeseries
                .filter(pl.col(grouping_col) == group)
                .sort(timepoint_col, descending=False)
                .to_pandas()
            )
            
            fig, ax = plt.subplots(figsize=(12,4))
            
            row = group_df.iloc[0].to_dict()
            title = make_title(row=row, well_col=well_col, meta_cols=meta_cols)
            
            sns.lineplot(
                data=group_df,
                x=timepoint_col,
                y=value_col,
                ax=ax,
                color='red',
                linewidth=2,
                markers=False,
                marker='o',
                markersize=2
            )
            
            ax.set_title(title)
            ax.set_xlabel("Timepoint")
            ax.set_ylabel("Expression")
            
            x_min = group_df[timepoint_col].min()
            x_max = group_df[timepoint_col].max()
            
            add_period_marks(ax=ax, x_min=x_min, x_max=x_max, interval=tick_interval)
            
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

In [53]:
out_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/enVision/enVision_visual_results/2026-06-01")
raw_timeseries = (
    pl.read_csv(raw_dataset, separator="\t")
    .join(plate_map, on="Well", how="left")
)

wells = raw_timeseries['Well'].unique().sort().to_list()
print(len(wells), wells)
display(raw_timeseries)

96 ['E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24', 'I01', 'I02', 'I03', 'I04', 'I05', 'I06', 'I07', 'I08', 'I09', 'I10', 'I11', 'I12', 'I13', 'I14', 'I15', 'I16', 'I17', 'I18', 'I19', 'I20', 'I21', 'I22', 'I23', 'I24', 'J01', 'J02', 'J03', 'J04', 'J05', 'J06', 'J07', 'J08', 'J09', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24']


Well,CalcResultI,Signal,Timepoint,MoleculeID,Human_Readable_Name,group_label,detected_period
str,i64,i64,f64,str,str,str,f64
"""E01""",20,4,0.0,null,null,"""nan | nan""",26.0
"""E02""",20,4,0.0,null,null,"""nan | nan""",26.0
"""E03""",20,4,0.0,null,null,"""nan | nan""",47.666667
"""E04""",20,4,0.0,null,null,"""nan | nan""",47.666667
"""E05""",60,12,0.0,null,null,"""nan | nan""",47.666667
…,…,…,…,…,…,…,…
"""J20""",40,8,138.0,null,null,"""nan | nan""",40.857143
"""J21""",40,8,138.0,null,null,"""nan | nan""",40.857143
"""J22""",80,16,138.0,null,null,"""nan | nan""",28.6


## Raw time series

In [ ]:
out_file = out_dir / "raw_timeseries.pdf"

value = "CalcResultI"
meta_cols = ['group_label' , 'detected_period']
timepoint_col = 'Timepoint'
well_col = "Well"

smoothed_raw = savgol_smooth(
    dataset=raw_timeseries.sort(
        ['group_label', well_col, timepoint_col],
        descending=[True, False, False]
        ),
    value_col=value,
    well_col=well_col,
    )

with PdfPages(out_file) as pdf:
    g = sns.FacetGrid(
        data=smoothed_raw,
        col="Well",
        col_wrap=12,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    
    g.map_dataframe(
        sns.lineplot,
        x=timepoint_col,
        y=value,
        color='red',
        linewidth=1.5
        )
    
    x_min = smoothed_raw[timepoint_col].min()
    x_max = smoothed_raw[timepoint_col].max()
    
    tick_marks = np.arange(np.ceil(x_min / 6) * 6, x_max, 6)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, desc='Well progress', total=len(wells)):
        well_df = (
            smoothed_raw
            .filter(pl.col(well_col) == well)
            .sort(timepoint_col, descending=False)
            .to_pandas()
        )
        
        fig, ax = plt.subplots(figsize=(12,4))
        
        row = well_df.iloc[0].to_dict()
        title = make_title(row=row, well_col=well_col, meta_cols=meta_cols)
        
        sns.lineplot(
            data=well_df,
            x=timepoint_col,
            y=value,
            ax=ax,
            color='red',
            linewidth=2,
            markers=False,
            marker='o',
            markersize=2
        )
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        
        add_period_marks(ax=ax, x_min=x_min, x_max=x_max, interval=6)
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        

Well progress: 100%|██████████| 96/96 [00:25<00:00,  3.73it/s]


## Detrended data

In [43]:
timepoint_col = "Timepoint"
well_coll = "Well"
feature = "CalcResultI"
value = "mean"
meta_cols = ['group_label' , 'detected_period']

detrended_timeseries = (
    pl.read_parquet(detrended_datast)
    .filter(pl.col('feature') == feature)
    .select([well_col, timepoint_col, value])
    .join(plate_map, on=well_col, how='left')
    .sort([timepoint_col, 'group_label'], descending=[False, False])
)

detrended_timeseries = savgol_smooth(
    dataset=detrended_timeseries,
    value_col=value,
    well_col=well_col
)

display(detrended_timeseries)

wells = detrended_timeseries[well_col].unique().to_list()

out_file = out_dir / "smoothed_detrended_timeseries._.28.6h.pdf"

with PdfPages(out_file) as pdf:
    g = sns.FacetGrid(
        data=detrended_timeseries,
        col='group_label',
        col_wrap=12,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    
    g.map_dataframe(
        sns.lineplot,
        x=timepoint_col,
        y=value,
        color='red',
        linewidth=1.5,
        errorbar='sd',
        err_kws={'alpha': 0.5, 'facecolor': 'red'}
        )
    
    x_min = detrended_timeseries[timepoint_col].min()
    x_max = detrended_timeseries[timepoint_col].max()
    tick_marks = np.arange(np.ceil(x_min / 6) * 6, x_max, 6)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc='Well Progress'):
        well_df = (detrended_timeseries
                   .filter(pl.col(well_col) == well)
                   .sort(timepoint_col)
                   .to_pandas()
                   )

        fig, ax = plt.subplots(figsize=(12,4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        sns.lineplot(
            data=well_df, 
            x=timepoint_col, 
            y=value, 
            ax=ax, 
            color='red',
            linewidth=1.5)
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        add_period_marks(
            ax=ax, 
            x_min=well_df[timepoint_col].min(), 
            x_max=well_df[timepoint_col].max(),
            interval=6)
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        
    

Well,Timepoint,mean,MoleculeID,Human_Readable_Name,group_label,detected_period
str,f64,f64,str,str,str,f64
"""F05""",30.0,-54.591168,null,null,"""nan | nan""",28.6
"""F05""",30.5,-26.083258,null,null,"""nan | nan""",28.6
"""F05""",31.0,-3.396757,null,null,"""nan | nan""",28.6
"""F05""",31.5,13.842243,null,null,"""nan | nan""",28.6
"""F05""",32.0,26.007649,null,null,"""nan | nan""",28.6
…,…,…,…,…,…,…
"""E23""",108.0,-5.418589,null,null,"""nan | nan""",47.666667
"""E23""",108.5,-3.537174,null,null,"""nan | nan""",47.666667
"""E23""",109.0,-1.070927,null,null,"""nan | nan""",47.666667


Well Progress: 100%|██████████| 96/96 [00:07<00:00, 12.44it/s]


## Group label (same as above)

In [ ]:
out_file = out_dir / "GROUPED._.smoothed_detrended_timeseries._.28.6h.pdf"

print(out_file.parent.exists())

plot_plates(
    timeseries=detrended_timeseries,
    timepoint_col=timepoint_col,
    well_col=well_col,
    grouping_col='group_label',
    value_col=value,
    meta_cols=meta_cols,
    out_file=out_file,
    smoothed=False,
    col_wrap=4,
    tick_interval=6
)

True


Well progress: 100%|██████████| 17/17 [00:33<00:00,  1.96s/it]


In [65]:
out_file = out_dir / "GROUPED._.raw_timeseries.pdf"

plot_plates(
    timeseries=raw_timeseries,
    timepoint_col=timepoint_col,
    well_col=well_col,
    grouping_col='group_label',
    value_col='CalcResultI',
    meta_cols=meta_cols,
    out_file=out_file,
    smoothed=True,
    col_wrap=4,
    tick_interval=6
)

Well progress: 100%|██████████| 17/17 [00:34<00:00,  2.00s/it]
